# Treino Grupo A (ResM) — Kaggle T4

**Autor**: Massanori  
**Data**: 18/05/2026  
**Descrição**: Orquestra smoke + treino completo do Grupo A (`ResidualMagnitudeModule`) no S5 do TCC.

## Pré-requisitos

- Settings → Accelerator = **T4 x1** (ou P100)
- Add Input → **tcc-mri-recons-varnet-brain-4x**

## Fluxo

1. Célula 2: clone do repo + instala dependencias
2. Célula 3: **diagnóstico** (GPU + estrutura do dataset). Falha cedo se algo estiver errado.
3. Célula 4: **smoke 500 iters** (gate antes do treino full, ~3-10 min em T4)
4. Célula 5: **treino completo 210k iters** (~12h GPU)
5. Célula 6: empacota outputs para virar Kaggle dataset `tcc-mri-resm-checkpoints`

In [ ]:
# Célula 2 — Setup: clone repo + dependencias
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/KR0N0S7/tcc-mri-uncertainty.git'
REPO_DIR = Path('/kaggle/working/tcc-mri-uncertainty')

if REPO_DIR.is_dir():
    print(f'Repo ja existe em {REPO_DIR}; git pull...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# HEAD do repo para auditoria
print('\nHEAD do repo:')
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=False)

print('\nInstalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(REPO_DIR / 'requirements.txt'),
    'python-dotenv',
], check=True)
print('Setup OK')

In [ ]:
# Célula 3 — Diagnóstico OBRIGATÓRIO. NAO prossiga em caso de falha.
#
# Auto-detecta o dataset em 3 padrões comuns de path no Kaggle:
#   1. /kaggle/input/<slug>/                    (padrão documentado)
#   2. /kaggle/input/datasets/<user>/<slug>/    (observado em datasets privados)
#   3. /kaggle/input/<slug>/<subdir>/           (zip auto-extraido com 1 nivel extra)
import os
import subprocess
import sys
from pathlib import Path

DATASET_SLUG = 'tcc-mri-recons-varnet-brain-4x'
EXPECTED_SUBS = {'train', 'val', 'cal', 'test'}


def find_dataset_root(slug):
    """Procura o dataset nos 3 padroes comuns de path Kaggle.

    Retorna o Path que tem {train, val, cal, test} como filhos diretos,
    ou levanta FileNotFoundError com diagnostico.
    """
    candidates = []

    # Padrao 1: /kaggle/input/<slug>/
    candidates.append(Path('/kaggle/input') / slug)

    # Padrao 2: /kaggle/input/datasets/<user>/<slug>/
    datasets_root = Path('/kaggle/input/datasets')
    if datasets_root.is_dir():
        for user_dir in datasets_root.iterdir():
            if user_dir.is_dir():
                candidates.append(user_dir / slug)

    print(f'Candidatos a investigar:')
    for c in candidates:
        print(f'  {c} (existe: {c.is_dir()})')

    # Testa cada candidato: direto OU com 1 nivel extra de aninhamento
    for cand in candidates:
        if not cand.is_dir():
            continue
        subs = {p.name for p in cand.iterdir() if p.is_dir()}
        if EXPECTED_SUBS.issubset(subs):
            return cand
        # Aninhamento de 1 nivel (e.g. zip auto-extraido)
        sub_dirs = [p for p in cand.iterdir() if p.is_dir()]
        if len(sub_dirs) == 1 and (sub_dirs[0] / 'train').is_dir():
            print(f'  Aninhamento extra detectado em {cand}; usando {sub_dirs[0]}')
            return sub_dirs[0]

    raise FileNotFoundError(
        f'Dataset "{slug}" nao encontrado em nenhum dos candidatos acima. '
        f'Verifique se foi adicionado como Input (Add Data) no painel direito '
        f'do notebook. Veja o ls de /kaggle/input/ abaixo para inspecionar.'
    )


print('=' * 60)
print('DIAGNOSTICO')
print('=' * 60)

# 1. O que tem em /kaggle/input/ (visao geral)
print('\n[1] Conteudo de /kaggle/input/:')
subprocess.run(['ls', '-la', '/kaggle/input/'], check=False)
if Path('/kaggle/input/datasets').is_dir():
    print('\n    Conteudo de /kaggle/input/datasets/:')
    subprocess.run(['ls', '-la', '/kaggle/input/datasets/'], check=False)

# 2. Auto-detecta a raiz real do dataset
print(f'\n[2] Procurando dataset com slug "{DATASET_SLUG}":')
actual_root = find_dataset_root(DATASET_SLUG)
print(f'\n  -> RECONS_ROOT resolvido: {actual_root}')
subprocess.run(['ls', '-la', str(actual_root)], check=False)

# 3. Verifica subpastas obrigatorias com contagem
print('\n[3] Subpastas do dataset:')
for sub in ('train', 'val', 'cal', 'test'):
    p = actual_root / sub
    if not p.is_dir():
        raise FileNotFoundError(f'Subpasta ausente: {p}')
    n_npz = len(list(p.glob('*.npz')))
    print(f'    {sub}: {n_npz} arquivos .npz')

# 4. GPU — critico: treino sem GPU e impraticavel
import torch
print(f'\n[4] CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU nao habilitada. Va em Settings -> Accelerator -> T4 x1 '
        'e CLIQUE em Save. Depois reinicie o kernel (Run > Restart).'
    )
print(f'    GPU: {torch.cuda.get_device_name(0)}')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'    VRAM: {vram_gb:.1f} GB')

# 5. Env vars para o src.config
os.environ['TCC_RECONS_DIR'] = str(actual_root)
os.environ['TCC_ANOTADOS_DIR'] = '/kaggle/working/_dummy_anotados'
os.environ['TCC_BRAIN_CSV'] = '/kaggle/working/_dummy_brain.csv'
Path('/kaggle/working/_dummy_anotados').mkdir(exist_ok=True)
Path('/kaggle/working/_dummy_brain.csv').touch()

print('\n[5] Config do projeto:')
subprocess.run([sys.executable, '-m', 'src.config'], check=True)

# Exporta para celulas seguintes
RECONS_ROOT = str(actual_root)

print('\n' + '=' * 60)
print('DIAGNOSTICO OK — seguro prosseguir')
print('=' * 60)

In [ ]:
# Célula 4 — Smoke: 500 iters do Grupo A.
# Gate antes do treino completo. Esperado: ~3-10 min em T4, razao final/inicial < 0.9.
import subprocess
import sys

smoke_run = '/kaggle/working/runs/group_a_smoke'

result = subprocess.run([
    sys.executable, 'scripts/smoke_train_resm.py',
    '--device', 'cuda',
    '--n-iters', '500',
    '--recons-dir', RECONS_ROOT,
    '--run-dir', smoke_run,
])

assert result.returncode == 0, (
    'SMOKE FALHOU. Veja log acima. NAO prossiga para o treino completo. '
    'Se loss nao decresce em 500 iters, ha bug a corrigir antes de gastar GPU.'
)
print('\nSMOKE OK — seguro rodar o treino completo na proxima celula.')

In [ ]:
# Célula 5 — Treino completo: 210000 iters do Grupo A.
# T4: ~12h. Kaggle session max = 12h, GPU quota = 30h/semana.
# Resume automatico em run_dir/last.pt se a sessao cair.
#
# Se este notebook for executado novamente (continuacao em nova sessao),
# basta rerodar todas as celulas — a Celula 5 retoma de onde parou
# automaticamente atraves de scripts/train_resm.py.
import subprocess
import sys

full_run = '/kaggle/working/runs/group_a_full'

result = subprocess.run([
    sys.executable, 'scripts/train_resm.py',
    '--device', 'cuda',
    '--recons-dir', RECONS_ROOT,
    '--run-dir', full_run,
    '--num-workers', '2',
])

assert result.returncode == 0, 'Treino falhou. Verifique log e last.pt.'
print('\nTreino completo concluido. Checkpoints em', full_run)

In [ ]:
# Célula 6 — Empacota outputs para virar Kaggle dataset.
# Apos rodar isso, va no painel direito (Output > Save Version > New Dataset).
# O dataset resultante (`tcc-mri-resm-checkpoints`) sera input para:
#   - Grupo B (comparação)
#   - Grupo C (comparação)
#   - Calibracao conforme S5.7 (sanity check do baseline)
import shutil
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/tcc-mri-resm-checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)

src_dir = Path(full_run)
copied = []
for fname in ('last.pt', 'best.pt', 'metrics.csv', 'config_snapshot.json'):
    src = src_dir / fname
    if src.exists():
        shutil.copy2(src, OUTPUT_DIR / fname)
        size_mb = src.stat().st_size / 1e6
        copied.append((fname, size_mb))
        print(f'  copiado: {fname} ({size_mb:.1f} MB)')
    else:
        print(f'  AUSENTE (pulando): {fname}')

print(f'\nOutput pronto em {OUTPUT_DIR}')
print('Salve como Kaggle dataset com slug: tcc-mri-resm-checkpoints')

## Próximos passos

Após este notebook concluir e o dataset `tcc-mri-resm-checkpoints` estar publicado:

### `notebooks/kaggle_train_groupB.ipynb` (Grupo B — QR, replicação do paper base)

Mesma estrutura deste notebook, mas:

- `ResidualMagnitudeModule` → `QuantileRegressionModule`
- `resm_loss` → `qr_loss`
- `loss_kwargs={'alpha': 0.10}` no treino
- Smoke + treino completo (~12h GPU para 210k iters; QR tem ~2x params, mas o gargalo é o data loader, não compute)
- **Sanity check obrigatório pós-treino**: Pearson(uncertainty média, |recon−target| média) por slice deve dar **~0,91** (Giannakopoulos et al., 2026, Tabela 1)

### `notebooks/kaggle_train_groupC.ipynb` (Grupo C — QR-Lesion, contribuição original)

Quase idêntico ao Grupo B, mas:

- `qr_loss` → `qr_lesion_loss`
- `loss_kwargs={'alpha': 0.10, 'lambda_lesion': 5.0}`
- `masks_dir` adicionado ao `ReconsSliceDataset` (do S3)

### Calibração conforme (S5.7)

Notebook separado que carrega os 3 checkpoints e roda no split `cal` (Romano et al., 2019; Angelopoulos & Bates, 2023).